# PPO Sweep — Shard Runner

Runs a subset of the 35 PPO cells (7 presets x 5 seeds) at 100k timesteps.
Open 5 copies of this notebook in parallel, each with a different SHARD_ID.

**Setup**: Runtime -> Change runtime type -> A100 GPU (or T4 if A100 unavailable).

**Parallelism**: 5 shards x 7 cells each = 35 total. Each shard ~14-22h wall.
Use `--limit-time 72000` (20h) to stay within Colab's 24h session limit.

In [ ]:
# === CONFIG: change SHARD_ID for each parallel session ===
SHARD_ID = 0       # 0, 1, 2, 3, or 4
TOTAL_SHARDS = 5
TOTAL_TIMESTEPS = 100_000
LIMIT_TIME_S = 72000  # 20h safety margin within 24h session
print(f'Shard {SHARD_ID}/{TOTAL_SHARDS}, timesteps={TOTAL_TIMESTEPS}, limit={LIMIT_TIME_S}s')

In [ ]:
# Cell 1: Mount Drive + unzip bundle
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
DRIVE_ROOT = '/content/drive/MyDrive/arcgis-farmland-mpc'
BUNDLE = f'{DRIVE_ROOT}/ppo_sweep_bundle.zip'
assert os.path.exists(BUNDLE), f'Upload ppo_sweep_bundle.zip to {DRIVE_ROOT}'
!unzip -q -o $BUNDLE -d /content
!ls /content/src

In [ ]:
# Cell 2: Install deps
!pip install -q sb3-contrib geopandas libpysal opensimplex rasterio pyyaml shapely

In [ ]:
# Cell 3: Env vars + GPU check
import os, subprocess
os.environ['TEST_SRC_ROOT'] = '/content/src/test'
os.environ['ADK_SRC_ROOT']  = '/content/src/adk'
print(subprocess.check_output(['nvidia-smi'], text=True)[:500])

In [ ]:
# Cell 4: Generate all 35 datasets (shared across shards, ~30 min)
# If datasets already exist on Drive, copy them instead.
%cd /content/src/benchmark
import pathlib
presets = ['bishan_clone','neijiang_clone','plain_small_cons','plain_large_cons',
           'plain_medium_frag','mixed_medium_frag','hilly_small_cons']
for p in presets:
    for s in range(5):
        out = f'data_dev/{p}_seed{s}'
        if pathlib.Path(f'{out}/manifest.json').exists():
            print(f'skip {out}')
            continue
        !python -m generator.generate --preset presets/{p}.yaml --seed {s} --out {out}
print('Done: datasets generated')

In [ ]:
# Cell 5: Run PPO sweep for this shard
%cd /content/src/benchmark
!python -m sweep.runner \
  --manifest sweep_manifest.csv \
  --state sweep_state_shard{SHARD_ID}.json \
  --data-root data_dev \
  --results-root results \
  --only-method PPO \
  --shard {SHARD_ID}/{TOTAL_SHARDS} \
  --resume \
  --limit-time {LIMIT_TIME_S}

In [ ]:
# Cell 6: Package results + state for download
%cd /content/src/benchmark
import pathlib, shutil
out_zip = f'/content/ppo_results_shard{SHARD_ID}.zip'
!cd /content/src/benchmark && zip -r {out_zip} results/ sweep_state_shard{SHARD_ID}.json
# Also copy to Drive for persistence
drive_out = f'{DRIVE_ROOT}/ppo_results_shard{SHARD_ID}.zip'
shutil.copy(out_zip, drive_out)
print(f'Results saved to {drive_out}')
print(f'Also available at {out_zip} for direct download')